# Training data explorer

Inspect `data/processed/training_rows.parquet` — the 21-feature training rows
built from 90 days of Binance BTCUSDT ticks, sampled at sec 60..840 within
each 15-minute window.

Helpers live in `notebooks/utils.py`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / "src").is_dir() is False:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from notebooks.utils import (
    set_display, load, head, window, windows_head,
    stats, label_balance, feature_label_corr, summary,
    feature_cols, META_COLS,
)

set_display()
df = load()
summary(df)

## 1 · Head

In [ ]:
head(df, n=15)

In [ ]:
# Just a few columns, if the full table is too wide.
head(df, n=15, cols=["body_pct", "flow_imbalance_60s", "volume_pace", "green_ratio", "session_bucket"])

## 2 · One window across all 14 sample points

See how the feature vector evolves as the 15-min candle progresses.

In [ ]:
window(df, idx=5000)   # arbitrary chronological window

In [ ]:
# Or pick by timestamp:
# window(df, ts="2026-04-15 12:00")

In [ ]:
windows_head(df, n=2, start_idx=0)   # first 2 windows = 28 rows

## 3 · Feature stats

In [ ]:
stats(df)

## 4 · Label balance

Overall and broken down by session / sample_sec / day-of-week.

In [ ]:
label_balance(df)

In [ ]:
label_balance(df, by="session_bucket")
# 0=Asia dead zone, 1=Asia, 2=London, 3=US, 4=US-PM

In [ ]:
label_balance(df, by="day_of_week")
# 0=Mon ... 6=Sun

In [ ]:
label_balance(df, by="sample_sec")
# Should be flat — same label assigned to all 14 samples of a window

## 5 · Feature → label correlation

Quick read on which features carry signal. `body_pct` and `body_time_product`
should dominate — they're a near-direct read on the eventual outcome late
in the candle. More interesting: flow features and 1-min sequence features
that carry signal *early*.

In [ ]:
feature_label_corr(df)

In [ ]:
# How does each feature's predictive power change as the candle progresses?
feature_label_corr(df, by_sample_sec=True)

## Scratch

Cells below for ad-hoc exploration.

In [ ]:
# df.query('flow_imbalance_60s > 0.5 and sample_sec == 60')[['window_open','body_pct','label']].head()